# Wavelet Feature SNR Robustness Analysis

This notebook compares RecordName-aligned Wavelet feature CSVs with a clean feature reference. It does not load labels or train any model. Configure paths below, then run all cells.

In [ ]:
# ============================== CONFIG ==============================
from pathlib import Path

# Set this to the cloned ECG project. Mount Drive first if the project/data live there.
PROJECT_DIR = Path('/content/ecg_ptbxl_benchmarking')
DATA_ROOT = Path('/content/drive/MyDrive/ECG/wavelet_features')

# Required clean reference CSV. All normalisation statistics come only from this file.
CLEAN_FEATURE_CSV = DATA_ROOT / 'clean_wavelet_features.csv'

# Configure only files you want analysed. Set a mapping value to None, or omit it, when unavailable.
NOISY_FEATURE_CSVS = {
    '24': DATA_ROOT / 'noisy_24db_wavelet_features.csv',
    '12': DATA_ROOT / 'noisy_12db_wavelet_features.csv',
    '6': DATA_ROOT / 'noisy_6db_wavelet_features.csv',
    '0': DATA_ROOT / 'noisy_0db_wavelet_features.csv',
    '-6': DATA_ROOT / 'noisy_minus6db_wavelet_features.csv',
}
DENOISED_FEATURE_CSVS = {
    '24': DATA_ROOT / 'denoised_24db_wavelet_features.csv',
    '12': DATA_ROOT / 'denoised_12db_wavelet_features.csv',
    '6': DATA_ROOT / 'denoised_6db_wavelet_features.csv',
    '0': DATA_ROOT / 'denoised_0db_wavelet_features.csv',
    '-6': DATA_ROOT / 'denoised_minus6db_wavelet_features.csv',
}

OUTPUT_DIR = DATA_ROOT / 'wavelet_snr_robustness_results'
ENABLE_BOOTSTRAP_CI = True
BOOTSTRAP_ITERATIONS = 1000
PRESERVATION_THRESHOLDS = [0.1, 0.25, 0.5, 1.0]
RANDOM_SEED = 42
EPSILON = 1e-8

# 'strict' requires identical RecordName values. Use 'prefix' only for an explicit
# one-to-one ID rule, e.g. 00001_lr from 00001_lr_mixed_snr6.mat. Directories
# are accepted and their CSV chunks are concatenated before validation.
RECORD_ID_MODE = 'strict'
RECORD_PREFIX_PATTERN = r'^(\d+_lr)'
# =====================================================================

In [ ]:
# Optional when Drive is not already mounted:
# from google.colab import drive
# drive.mount('/content/drive')

import importlib.util
import sys
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f'PROJECT_DIR does not exist: {PROJECT_DIR}. Clone/upload this repository or update CONFIG.'
    )
# Load by file path because this repository's `code` package collides with Python's
# standard-library module of the same name in some notebook runtimes.
module_path = PROJECT_DIR / 'code' / 'wavelet_feature_snr_robustness.py'
spec = importlib.util.spec_from_file_location('wavelet_feature_snr_robustness', module_path)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)
run_analysis = module.run_analysis

config = {
    'data_root': str(DATA_ROOT),
    'clean_feature_csv': str(CLEAN_FEATURE_CSV),
    'noisy_feature_csvs': {snr: str(path) if path else None for snr, path in NOISY_FEATURE_CSVS.items()},
    'denoised_feature_csvs': {snr: str(path) if path else None for snr, path in DENOISED_FEATURE_CSVS.items()},
    'output_dir': str(OUTPUT_DIR),
    'enable_bootstrap_ci': ENABLE_BOOTSTRAP_CI,
    'bootstrap_iterations': BOOTSTRAP_ITERATIONS,
    'preservation_thresholds': PRESERVATION_THRESHOLDS,
    'random_seed': RANDOM_SEED,
    'epsilon': EPSILON,
    'record_id_mode': RECORD_ID_MODE,
    'record_prefix_pattern': RECORD_PREFIX_PATTERN,
}
results = run_analysis(config)

In [ ]:
# Concise final table. Full tables, figures, and summary.md are in OUTPUT_DIR.
display(results['overall'].sort_values(['dataset_type', 'snr'], key=lambda column: column.map(lambda value: float(str(value).replace('dB', '').strip())) if column.name == 'snr' else column))
print(f'Outputs saved to: {OUTPUT_DIR}')